In [1]:
from Preprocess import EEGPreprocessor
import os
from tqdm import tqdm  
import numpy as np
import pandas as pd
import shutil

# Configuration
config={
    'root' : '/teamspace/studios/this_studio/Dataset/ds004504/',
    'subject_path' : '/teamspace/studios/this_studio/phdResearch/data1n/Subjects',
    'label_path' : '/teamspace/studios/this_studio/phdResearch/data1n/Labels',
    ##############

    'epoch_duration' : 5.0,   # second
    'epoch_overlap' : 0.0,    # No overlap
    'resample_freq' : 128,    # HZ point per second
    'l_freq' : 0.5,
    'h_freq' : 40.0,
    'notch_freq' : 50.0,
    'asr_cutoff' : 20,
    'iclabel_threshold' : 0.90,
    'random_state' : 42,
    'use_pyprep' : False,
    'flat_std_thresh' : 1.5e-6,  
    'verbose' : False,
    
    ##############
}

if os.path.exists(config['subject_path']):
    shutil.rmtree(config['subject_path'])
    shutil.rmtree(config['label_path'])
if not os.path.exists(config['subject_path']):
    os.makedirs(config['subject_path'])
if not os.path.exists(config['label_path']):
    os.makedirs(config['label_path'])
    
#####################################################################################
# Define data files
AD_data = [config['root'] + f"sub-{i+1:03}/eeg/sub-{i+1:03}_task-eyesclosed_eeg.set" 
           for i in range(36)]
HC_data = [config['root'] + f"sub-{i+37:03}/eeg/sub-{i+37:03}_task-eyesclosed_eeg.set" 
           for i in range(29)]


all_files = AD_data + HC_data 
label_list = []

# Process with progress bar
successful = 0
failed = 0
sub_id = 1
for file_path in tqdm(all_files, desc="Processing EEG files"):
    if os.path.exists(file_path):
        try:
            # Initialize preprocessor
            preprocessor = EEGPreprocessor(
                           epoch_duration = config['epoch_duration'],
                           epoch_overlap = config['epoch_overlap'],
                           resample_freq = config['resample_freq'],
                           l_freq = config['l_freq'],
                           h_freq = config['h_freq'],
                           notch_freq = config['notch_freq'],
                           asr_cutoff = config['asr_cutoff'],
                           iclabel_threshold = config['iclabel_threshold'],
                           random_state = config['random_state'],
                           flat_std_thresh = config['flat_std_thresh'],  
                           verbose = config['verbose']
                          )                    
            
            # Process file
            data = preprocessor.preprocess(file_path)
            
            print(data.shape)
            np.save(os.path.join(config['subject_path'], f'Sub_{sub_id:03d}.npy'), data)
            if sub_id <=36:
                label_list.append(np.array([sub_id,1]))
            else:
                label_list.append(np.array([sub_id,0]))
            sub_id += 1
            successful += 1
            print('\n')
            
        except Exception as e:
            print(f" Error in file {file_path}: {str(e)}")
            import traceback
            traceback.print_exc() 
            failed += 1
            continue
    
    print("-------------------------------------\n")
    print(f"Processing complete")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    
labels = np.array(label_list)

labels_df = pd.DataFrame(labels, columns=['subject_id', 'label'])
labels_df['subject_id'] = [f'Sub_{i:03d}' for i in labels_df['subject_id']]
condition_mapping = {0: 'HC', 1: 'AD'}
labels_df['Group'] = labels_df['label'].map(condition_mapping)
csv_path = os.path.join(config['label_path'], 'labels.csv')
labels_df.to_csv(csv_path, index=False)
print(f"Labels saved to: {csv_path}")


Processing EEG files:   0%|          | 0/65 [00:00<?, ?it/s]


[Pipeline Start] -> /teamspace/studios/this_studio/Dataset/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
-> Standard 10-20 EEG detected
-> Band-pass filtering
-> Removing line noise
-> Resampling to 128 Hz
-> PyPREP disabled.
-> Selecting optimal ASR calibration window...
[Success] Optimal ASR baseline selected at 25th percentile: 384.00s to 414.00s
(19, 3969)
---------------------
Shape: (19, 3969)
dtype: float64
NaN: 0
Inf: 0
Samples: 3969
---------------------
-> Variance ratio : 0.943
-> eeg_reference average 
-> Fitting ICA...
-> Running ICLabel...


Processing EEG files:   2%|▏         | 1/65 [00:12<13:18, 12.48s/it]


ICLabel Classification
IC 00 | eye blink          | 0.893
IC 01 | eye blink          | 0.996
IC 02 | eye blink          | 0.643
IC 03 | brain              | 1.000
IC 04 | brain              | 0.935
IC 05 | brain              | 0.989
IC 06 | eye blink          | 0.931
IC 07 | eye blink          | 0.583
IC 08 | brain              | 0.606
IC 09 | brain              | 0.673
IC 10 | brain              | 0.992
IC 11 | brain              | 0.956
IC 12 | other              | 0.802
IC 13 | brain              | 0.955
IC 14 | other              | 0.507
IC 15 | muscle artifact    | 0.643
IC 16 | brain              | 0.986
IC 17 | brain              | 0.772
Removing 2 ICs
Harmonized channels count: 19/19
[Pipeline Success]

========== QC REPORT ==========
Channels           : 19
Sampling Rate      : 128.0 Hz
ASR Variance Ratio : 0.943
Removed ICs        : 2
Epochs             : 119
Output Shape       : (119, 19, 640)
(119, 19, 640)


-------------------------------------

Processing complete
Succe

Processing EEG files:   3%|▎         | 2/65 [00:28<15:31, 14.79s/it]

...


ICLabel Classification
IC 00 | eye blink          | 0.979
IC 01 | brain              | 0.999
IC 02 | eye blink          | 0.806
IC 03 | other              | 0.831
IC 04 | brain              | 1.000
IC 05 | brain              | 0.999
IC 06 | other              | 0.948
IC 07 | brain              | 0.995
IC 08 | brain              | 0.988
IC 09 | brain              | 0.650
IC 10 | muscle artifact    | 0.621
IC 11 | brain              | 0.982
IC 12 | brain              | 0.988
IC 13 | brain              | 0.777
IC 14 | eye blink          | 0.426
IC 15 | brain              | 0.729
IC 16 | brain              | 0.810
IC 17 | brain              | 0.661
Removing 1 ICs
Harmonized channels count: 19/19
[Pipeline Success]

========== QC REPORT ==========
Channels           : 19
Sampling Rate      : 128.0 Hz
ASR Variance Ratio : 1.000
Removed ICs        : 1
Epochs             : 161
Output Shape       : (161, 19, 640)
(161, 19, 640)


-------------------------------------

Processing complete
Succe

Processing EEG files:  98%|█████████▊| 64/65 [16:26<00:14, 14.46s/it]


ICLabel Classification
IC 00 | brain              | 1.000
IC 01 | brain              | 1.000
IC 02 | brain              | 1.000
IC 03 | eye blink          | 0.974
IC 04 | brain              | 0.994
IC 05 | eye blink          | 0.993
IC 06 | brain              | 0.995
IC 07 | brain              | 0.990
IC 08 | brain              | 0.943
IC 09 | brain              | 0.994
IC 10 | brain              | 0.983
IC 11 | brain              | 0.921
IC 12 | brain              | 0.982
IC 13 | brain              | 0.928
IC 14 | brain              | 0.989
IC 15 | brain              | 0.760
IC 16 | brain              | 0.855
IC 17 | muscle artifact    | 0.593
Removing 2 ICs
Harmonized channels count: 19/19
[Pipeline Success]

========== QC REPORT ==========
Channels           : 19
Sampling Rate      : 128.0 Hz
ASR Variance Ratio : 0.377
Removed ICs        : 2
Epochs             : 169
Output Shape       : (169, 19, 640)
(169, 19, 640)


-------------------------------------

Processing complete
Succe

Processing EEG files: 100%|██████████| 65/65 [16:40<00:00, 15.39s/it]


ICLabel Classification
IC 00 | eye blink          | 0.922
IC 01 | eye blink          | 0.605
IC 02 | brain              | 0.745
IC 03 | brain              | 0.999
IC 04 | brain              | 0.724
IC 05 | brain              | 0.997
IC 06 | brain              | 0.945
IC 07 | muscle artifact    | 0.990
IC 08 | muscle artifact    | 0.970
IC 09 | brain              | 0.358
IC 10 | brain              | 0.687
IC 11 | brain              | 0.970
IC 12 | brain              | 0.974
IC 13 | brain              | 0.997
IC 14 | brain              | 0.931
IC 15 | brain              | 0.960
IC 16 | brain              | 0.990
IC 17 | brain              | 0.996
Removing 3 ICs
Harmonized channels count: 19/19
[Pipeline Success]

========== QC REPORT ==========
Channels           : 19
Sampling Rate      : 128.0 Hz
ASR Variance Ratio : 0.336
Removed ICs        : 3
Epochs             : 176
Output Shape       : (176, 19, 640)
(176, 19, 640)


-------------------------------------

Processing complete
Succe